In [ ]:
# Un ranking temporal permite detectar los temas que ganan relevancia en el corpus.

from pathlib import Path
import json
import sqlite3

import pandas as pd

ACTIVITY_DIR = next(candidate for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (candidate / "data").is_dir() and (candidate / "submission").is_dir())
DATABASE_FILE = ACTIVITY_DIR / "data" / "scopus_proptech.db"
SUBMISSION_DIR = ACTIVITY_DIR / "submission"

In [ ]:
# document_keywords → documents
#       ↓                  ↓
#    keywords            years

database = sqlite3.connect(DATABASE_FILE)

In [ ]:
# Las CTE separan el conteo anual del ranking que permite comparar temas.

keyword_rankings = pd.read_sql_query(
    """
    WITH keyword_years AS (
        SELECT
            d.publication_year,
            k.keyword,
            COUNT(DISTINCT d.document_id) AS document_count
        FROM documents AS d
        JOIN document_keywords AS dk ON d.document_id = dk.document_id
        JOIN keywords AS k ON dk.keyword = k.keyword
        WHERE d.publication_year >= 2020
        GROUP BY d.publication_year, k.keyword
    ),
    ranked_keywords AS (
        SELECT
            publication_year,
            keyword,
            document_count,
            ROW_NUMBER() OVER (
                PARTITION BY publication_year
                ORDER BY document_count DESC, keyword
            ) AS rank_in_year
        FROM keyword_years
    )
    SELECT publication_year, keyword, document_count, rank_in_year
    FROM ranked_keywords
    WHERE rank_in_year <= 10
    ORDER BY publication_year, rank_in_year
    """,
    database,
)
keyword_rankings.head()

In [ ]:
# El archivo permite que una visualización o dashboard reutilice el ranking.

keyword_rankings.to_csv(SUBMISSION_DIR / "top_keywords_by_year.csv", index=False)
questions = [{"pregunta": "¿Qué palabras clave están entre las diez más frecuentes por año desde 2020?", "archivo_respuesta": "top_keywords_by_year.csv"}]
(SUBMISSION_DIR / "questions.json").write_text(json.dumps(questions, ensure_ascii=False, indent=2), encoding="utf-8")
database.close()